In [1]:
import warnings
warnings.simplefilter(action='ignore')

import requests, joblib, os
import numpy, pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import Lasso
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import TweedieRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestCentroid

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score

In [2]:
test_c =pandas.read_csv('/kaggle/input/crypto-data-2/test.csv')

In [3]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

for i in zip(test_c.columns, test_c.dtypes):
    if i[1]=='O':
        test_c[i[0]] = test_c[i[0]].fillna('Unknown')
        test_c[i[0]] = normalize.fit_transform(encoder.fit_transform(test_c[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        test_c[i[0]].fillna(test_c[i[0]].mean(), inplace=True)
        test_c[i[0]] = normalize.fit_transform(numpy.array(test_c[i[0]]).reshape(-1,1))

In [4]:
test_c = test_c.drop(columns=['label'])

In [5]:
class model_infrence:
    def __new__(self, test_c):
        self.X_train = pandas.read_csv('/kaggle/input/crypto-models/data/X_train.csv')
        self.y_train = pandas.read_csv('/kaggle/input/crypto-models/data/y_train.csv')
        self.X_test = pandas.read_csv('/kaggle/input/crypto-models/data/X_test.csv')
        self.y_test = pandas.read_csv('/kaggle/input/crypto-models/data/y_test.csv')
        self.test_c = test_c
        model_path = '/kaggle/input/crypto-models/model'

        self.model_m = {i : joblib.load(f'{model_path}/{i}') for i in os.listdir(model_path) if i != 'STACK.pkl'}
        results = self.manual_ensemble(self)# {'STACK' : model_s_pre.reshape(-1)} | 
        #print(results)
        for i in results.keys():
            submission = pandas.DataFrame({
                'ID': [i for i in range(1, len(self.test_c)+1)],
                'prediction': results[i],
            })
            submission.to_csv(f'{i}.csv', index = False)
        
    def manual_ensemble(self):
        weight, new_train_data, new_test_x_data, new_test_data, total = {}, {}, {}, {}, 0
        
        for i in self.model_m.keys():
            if i != 'KMeans':
                pre = self.model_m[i].predict(self.X_test)
                loss = mean_squared_error(self.y_test, pre)
                weight[i] = 1 - loss#((1 - loss) * 0.9)#//20)-0.22)*100
                if weight[i] > 0:#loss < 1:
                    total += weight[i]
                    print(f'RMSE LOSS {i} : {loss} WEIGHT : {weight[i]}')
                
                    new_train_data[i] = (self.model_m[i].predict(self.X_train) * weight[i]).reshape(-1)
                    new_test_x_data[i] = (pre * weight[i]).reshape(-1)
                    new_test_data[i] = (self.model_m[i].predict(self.test_c) * weight[i]).reshape(-1)
        
        self.new_train_data = pandas.DataFrame(new_train_data)
        self.new_test_x_data = pandas.DataFrame(new_test_x_data)
        self.new_test_data = pandas.DataFrame(new_test_data)
        
        model = ElasticNetCV()
        model.fit(self.new_train_data, self.y_train)
        
        manual = numpy.array(sum(list(new_test_data.values()))/total)
        manual_i = numpy.array(sum(list(new_test_x_data.values()))/total)
                
        print(f'RMSE SCORE MANUAL WEIGHTED ENSEMBLE : {100*(1-mean_squared_error(self.y_test, model.predict(self.new_test_x_data)))}')
        print(f'RMSE SCORE MANUAL WEIGHTED AVERAGE : {100*(1-mean_squared_error(self.y_test, manual_i))}')
        
        return {'M_META_MODEL' : model.predict(self.new_test_data).reshape(-1), 'M_AVERAGE_MODEL' :  manual.reshape(-1)}

In [6]:
model_R = model_infrence(test_c)

RMSE LOSS random_forest.pkl : 0.8485858574777492 WEIGHT : 0.15141414252225083
RMSE LOSS GradientBoostingRegressor.pkl : 0.4460083898346447 WEIGHT : 0.5539916101653553
RMSE LOSS extra_trees.pkl : 0.8843250686444639 WEIGHT : 0.11567493135553608
RMSE SCORE MANUAL WEIGHTED ENSEMBLE : 68.41658719554505
RMSE SCORE MANUAL WEIGHTED AVERAGE : 45.03145461722408
